# Text Masking: De-identifying BalanceCorpus Transcripts

**Thursday Masking Day · Tilburg Multiscale Summerschool 2026 · Take-home notebook**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WimPouw/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/09_text_masking_balance_corpus.ipynb)

---

You have masked faces. You have suppressed voices.  
But what about the **words**?

Even in a controlled referential communication game — describe a word without using  
five forbidden words — participants sometimes:

- Name real people ("it's like what Popeye eats")
- Reference places ("I had this in Amsterdam last week")
- Reveal personal details ("my gran makes these on Sundays")

A transcript is a text document. Text documents have the same GDPR obligations as  
video. This notebook closes the loop on Thursday's masking arc:

```
Face  →  MaskAnyone        (sessions 1–2)
Voice →  audio masking     (session 5 / notebook 04)
Text  →  this notebook     (the third modality)
```

### What you will do

| Step | Task |
|------|------|
| 1 | Load a BalanceCorpus transcript (real or synthetic) |
| 2 | Find personally identifying information (NER + patterns) |
| 3 | Replace it — same logic as blurring a face |
| 4 | Check: does the research-relevant content survive? |
| 5 | Save a de-identified transcript + audit log |

## Setup

In [ ]:
import sys, re, json, collections, textwrap
!{sys.executable} -m pip install -q spacy plotly
!{sys.executable} -m spacy download en_core_web_sm -q

import spacy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import HTML, display as ipy_display

plt.rcParams.update({
    'figure.facecolor': '#161616', 'axes.facecolor': '#262626',
    'axes.edgecolor': '#525252', 'axes.labelcolor': '#c6c6c6',
    'xtick.color': '#6f6f6f', 'ytick.color': '#6f6f6f',
    'text.color': '#f4f4f4', 'grid.color': '#393939', 'grid.linestyle': '--',
})

PLOTLY_DARK = dict(
    paper_bgcolor='#161616', plot_bgcolor='#262626',
    font_color='#f4f4f4',
    legend=dict(bgcolor='rgba(38,38,38,0.8)'),
)

nlp = spacy.load('en_core_web_sm')
print('Ready.')

## 1 — Load a transcript

If you ran **notebook 08** and saved a Whisper transcript, point `TRANSCRIPT_PATH` to it.  
Otherwise we use a synthetic BalanceCorpus-style transcript that contains realistic  
examples of personal information leaking through a controlled task.

In [ ]:
TRANSCRIPT_PATH = None   # e.g. Path('text_deid_outputs/transcript_deidentified.txt')

# Synthetic BalanceCorpus transcript — realistic leakage examples
# Each speaker turn is one trial; participant IDs are the corpus codes
DEMO_TEXT = """
[Trial 12 — target: doughnut — clue-giver: participant 103 — condition: board]
Participant 103: Um, it's like a ring, kind of fried dough thing — my mum Sarah makes
them every Sunday in Leiden. Very sweet, covered in icing, you know the ones from
Dunkin' Donuts? Krispy Kreme? Like that. Homer Simpson eats them constantly.

[Trial 13 — target: spinach — clue-giver: participant 203 — condition: ground]
Participant 203: It's a dark green leafy vegetable. Popeye eats it. My flatmate —
her name's Yuki Tanaka — she put it in a smoothie this morning. Very healthy.
You find it at Albert Heijn. Tastes bitter raw but nice in pasta.

[Trial 14 — target: balloon — clue-giver: participant 103 — condition: board]
Participant 103: Round, you fill it with air or helium, they float — like at birthdays?
We had them at my sister Emma's party last week in Rotterdam. Red and yellow, very big.
Contact Emma at emma_103@gmail.com if you want details about the party.

[Trial 15 — target: tiger — clue-giver: participant 203 — condition: ground]
Participant 203: It's a large wild cat. Orange, black stripes. Found in India, Sumatra.
Our professor — Jan de Vries, Radboud University — showed us a documentary about them
last semester. They're endangered. Tony the Tiger is the cereal mascot.

[Trial 16 — target: hunger — clue-giver: participant 103 — condition: board]
Participant 103: Um, it's like — when you need food. A feeling. My phone number is
06-12345678 but anyway, it's a physical sensation, your stomach growls, you feel
kind of weak, you know, like before lunch.
"""

text = (Path(TRANSCRIPT_PATH).read_text(encoding='utf-8')
        if TRANSCRIPT_PATH else DEMO_TEXT)

print(f'Loaded: {len(text.split())} words')
print(textwrap.fill(text[:300].strip(), width=80), '...')

## 2 — What personal information is hiding in there?

Read the transcript above and note anything you would not want published:  
names, places, institutions, contact details.

Now let the model do the same job automatically.

In [ ]:
doc = nlp(text)

print('Named entities found by spaCy:')
print(f'{"Entity":<30} {"Label":<12} {"What it is"}')
print('-' * 65)

LABEL_DESC = {
    'PERSON': 'person name',
    'ORG':    'organisation',
    'GPE':    'city / country / region',
    'LOC':    'location',
    'FAC':    'facility / building',
    'EVENT':  'event',
    'WORK_OF_ART': 'creative work',
    'PRODUCT': 'product name',
    'DATE':   'date / time',
    'TIME':   'time expression',
}

SENSITIVE_LABELS = {'PERSON', 'ORG', 'GPE', 'LOC', 'FAC', 'EVENT'}

for ent in doc.ents:
    flag = '⚠' if ent.label_ in SENSITIVE_LABELS else ' '
    desc = LABEL_DESC.get(ent.label_, ent.label_)
    print(f'{flag} {ent.text:<28} {ent.label_:<12} {desc}')

## 3 — De-identify: replace entities and patterns

Two passes:
1. **NER-based** — replace each named entity with a type tag + counter  
2. **Pattern-based** — catch email, phone, URL that NER misses

In [ ]:
REPLACE_LABELS = {
    'PERSON': '[PERSON]',
    'ORG':    '[ORG]',
    'GPE':    '[PLACE]',
    'LOC':    '[PLACE]',
    'FAC':    '[PLACE]',
    'EVENT':  '[EVENT]',
}

PATTERNS = [
    (r'[\w.+-]+@[\w-]+\.[\w.]+',        '[EMAIL]'),
    (r'\+?[\d][\d\s\-().]{7,}[\d]',    '[PHONE]'),
    (r'https?://\S+|www\.\S+',          '[URL]'),
]

def deidentify(text: str):
    doc      = nlp(text)
    seen     = {}
    counters = collections.defaultdict(int)
    log      = []

    replacements = []
    for ent in doc.ents:
        if ent.label_ not in REPLACE_LABELS:
            continue
        base  = REPLACE_LABELS[ent.label_]
        canon = ent.text.strip().lower()
        if canon not in seen:
            counters[base] += 1
            seen[canon] = f'{base[:-1]}_{counters[base]}]'
        tag = seen[canon]
        replacements.append((ent.start_char, ent.end_char, ent.text, tag, ent.label_))
        log.append({'original': ent.text, 'tag': tag, 'label': ent.label_, 'method': 'NER'})

    result = text
    for start, end, _, tag, _ in sorted(replacements, key=lambda x: x[0], reverse=True):
        result = result[:start] + tag + result[end:]

    for pattern, tag in PATTERNS:
        for m in re.finditer(pattern, result):
            log.append({'original': m.group(), 'tag': tag, 'label': 'PATTERN', 'method': 'regex'})
        result = re.sub(pattern, tag, result)

    return result, log


text_deid, log = deidentify(text)
print(f'Replacements made: {len(log)}')
for entry in log:
    print(f"  [{entry['method']}] '{entry['original']}'  →  {entry['tag']}")

## 4 — Before / after view

In [ ]:
def highlight(text: str) -> str:
    return re.sub(
        r'(\[[A-Z_0-9]+\])',
        r'<mark style="background:#f1c21b;color:#161616;padding:1px 4px;'
        r'border-radius:2px;font-weight:bold">\1</mark>',
        text
    )

ipy_display(HTML(f"""
<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;
            font-family:monospace;font-size:11px">
  <div style="background:#262626;padding:14px;border:1px solid #393939">
    <div style="color:#6f6f6f;margin-bottom:8px;font-size:10px;
                letter-spacing:.1em">ORIGINAL</div>
    <div style="color:#f4f4f4;white-space:pre-wrap">{text.strip()}</div>
  </div>
  <div style="background:#262626;padding:14px;border:1px solid #393939">
    <div style="color:#6f6f6f;margin-bottom:8px;font-size:10px;
                letter-spacing:.1em">DE-IDENTIFIED</div>
    <div style="color:#f4f4f4;white-space:pre-wrap">{highlight(text_deid.strip())}</div>
  </div>
</div>
"""))

## 5 — Does the research content survive?

We care about the *clue words* — the content words the participant used to describe  
the target. Let's check that the key nouns, verbs, and adjectives are still present  
after de-identification.

In [ ]:
STOPWORDS = {
    'i','me','my','you','your','we','they','it','is','was','are','were',
    'be','been','have','has','had','do','does','did','a','an','the',
    'and','or','but','in','on','at','to','for','of','by','with','from',
    'that','this','like','kind','sort','just','also','very','so',
}

def content_words(text: str) -> set:
    # Strip [TAG_N] tokens before spaCy sees them — otherwise spaCy splits
    # [PERSON_1] into punctuation + word tokens and PERSON gets POS=NOUN.
    clean = re.sub(r'\[[A-Z_0-9]+\]', ' ', text)
    doc   = nlp(clean.lower())
    return {
        t.lemma_ for t in doc
        if t.pos_ in {'NOUN', 'VERB', 'ADJ'}
        and t.lemma_ not in STOPWORDS
        and len(t.lemma_) > 2
    }

orig_words = content_words(text)
deid_words = content_words(text_deid)

preserved = orig_words & deid_words
removed   = orig_words - deid_words

print(f'Content words in original:      {len(orig_words)}')
print(f'Content words in de-identified: {len(deid_words)}')
print(f'Preserved:                      {len(preserved)}  '
      f'({100*len(preserved)/max(len(orig_words),1):.0f}%)')
print(f'Removed (were identifying):     {len(removed)}')
print()
print('Removed:', sorted(removed))
print()
print('Sample preserved:', sorted(preserved)[:20])

In [ ]:
fig = px.pie(
    values=[len(preserved), len(removed)],
    names=[f'Preserved ({len(preserved)})', f'Removed ({len(removed)})'],
    color_discrete_sequence=['#42be65', '#da1e28'],
    title='Research content words — impact of de-identification',
    hole=0.35,
)
fig.update_traces(textinfo='percent+label', textfont_size=12)
fig.update_layout(**PLOTLY_DARK, height=380)
fig.show()

## 6 — What the model missed

NER is not perfect. Some identifying references slip through:
- Fictional characters used as clues ("Popeye", "Homer Simpson", "Tony the Tiger")
- Brand names ("Dunkin' Donuts", "Krispy Kreme", "Albert Heijn")
- Indirect identifiers ("my mum", "my flatmate", "last week")

The cell below shows what remains in the de-identified text that a  
careful human reviewer would still flag.

In [ ]:
# Check what named entities spaCy still finds in the de-identified text
doc_deid = nlp(text_deid)

residual = [
    (ent.text, ent.label_)
    for ent in doc_deid.ents
    if ent.label_ in SENSITIVE_LABELS
    and not ent.text.startswith('[')
]

if residual:
    print('Residual identifying entities (missed by NER):')
    for text_e, label in residual:
        print(f'  {label:<12} "{text_e}"')
else:
    print('No residual entities detected — NER caught everything in this text.')

print()
print('Manual review should also check for:')
print('  - Fictional characters used as clues (Popeye, Homer, Tony the Tiger)')
print('  - Brand names (Dunkin\' Donuts, Albert Heijn, Krispy Kreme)')
print('  - Indirect identifiers ("my sister", "my flatmate", "last week")')
print('  - Unique events that could identify the speaker')

## 7 — Save de-identified transcript and audit log

In [ ]:
OUT_DIR = Path('text_deid_outputs')
OUT_DIR.mkdir(exist_ok=True)

(OUT_DIR / 'balance_corpus_transcript_deid.txt').write_text(
    text_deid, encoding='utf-8'
)

audit = {
    'tool': 'spaCy en_core_web_sm + regex',
    'replacements': log,
    'residual_check': [{'text': t, 'label': l} for t, l in residual],
    'stats': {
        'total_replacements': len(log),
        'ner_replacements':     sum(1 for e in log if e['method'] == 'NER'),
        'pattern_replacements': sum(1 for e in log if e['method'] == 'regex'),
        'content_words_original':    len(orig_words),
        'content_words_deid':        len(deid_words),
        'content_preserved_pct':     round(100*len(preserved)/max(len(orig_words),1), 1),
    }
}
(OUT_DIR / 'balance_corpus_deid_audit.json').write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('Saved:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')

---
## The Thursday masking arc — complete

| Modality | Tool | What was hidden |
|----------|------|----------------|
| Face (video) | MaskAnyone | identity from appearance |
| Voice (audio) | Pyannote / noise | identity from voice |
| Text (transcript) | spaCy NER + regex | names, places, contact details |

The same principle applies across all three:  
**replace the identifying signal while preserving the research signal.**

In video: the gesture, the posture, the timing survive.  
In audio: the prosody, the rhythm, the turn-taking survive.  
In text: the clue words, the discourse structure, the strategy survive.

---
## Go further

- **Train a custom NER model** — add fictional characters (Popeye, Homer) and brands  
  (Krispy Kreme, Albert Heijn) as custom entity types using spaCy's training pipeline
- **Indirect identifier detection** — write rules for relational terms  
  ("my [KINSHIP]", "my flatmate") using spaCy's `Matcher`
- **Pseudonymisation** — instead of `[PERSON_1]`, replace with a consistent fake name  
  (e.g. "Alex") so the transcript still reads naturally for qualitative analysis
- **Cross-participant consistency** — if the same real person is mentioned in two  
  different transcripts, they should get the same tag number across both
- **Evaluate recall** — create a ground-truth list of identifying spans and measure  
  what percentage the automatic pipeline catches

### Connection to Thursday

- **Notebook 08** produced the transcripts this notebook de-identifies
- **Notebook 07** showed that the *clue words* (content preserved here) are what carry  
  the semantic signal the researcher needs